# LC 338 — Counting Bits
**Day 44 | Bit Manipulation | Easy**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** `dp[i] = dp[i >> 1] + (i & 1)`. Right-shift drops
the last bit (reuses a solved sub-problem); `& 1` adds 1 if that
dropped bit was set. Build the answer array in one pass.

</div>

## Official Problem Statement

Given an integer `n`, return an array `ans` of length `n + 1` such
that for each `i` (`0 <= i <= n`), `ans[i]` is the **number of 1's**
in the binary representation of `i`.

**Constraints:**
- `0 <= n <= 10^5`

**Follow up:**
- It is very easy to come up with a solution with a runtime of
  `O(n log n)`. Can you do it in linear time `O(n)`?
- Can you do it without using any built-in function (e.g.,
  `bin(x).count('1')`)?
- Can you do it in `O(1)` space (output array doesn't count)?

## What This Is Actually Asking

For every integer from 0 to n, count how many 1-bits are in its
binary form and return all counts as an array.

The naive solution calls `bin(i).count('1')` for each i, which is
O(n log n) because counting bits in i takes O(log i) time.

The optimal trick: every number i is just some smaller number
(`i >> 1`) with possibly one extra bit. So we can reuse already-
computed answers — classic DP with bit manipulation.

The recurrence `dp[i] = dp[i >> 1] + (i & 1)` builds the full
table in O(n) time and O(1) extra space.

## Walk Through an Example by Hand

`n = 5`  →  expected: `[0, 1, 1, 2, 1, 2]`

```
i=0: dp[0] = 0          (base case)

i=1: binary = 1
  i>>1 = 0  →  dp[0] = 0
  i&1  = 1  (last bit is 1)
  dp[1] = 0 + 1 = 1  ✓

i=2: binary = 10
  i>>1 = 1  →  dp[1] = 1
  i&1  = 0  (last bit is 0)
  dp[2] = 1 + 0 = 1  ✓

i=3: binary = 11
  i>>1 = 1  →  dp[1] = 1
  i&1  = 1  (last bit is 1)
  dp[3] = 1 + 1 = 2  ✓

i=4: binary = 100
  i>>1 = 2  →  dp[2] = 1
  i&1  = 0  (last bit is 0)
  dp[4] = 1 + 0 = 1  ✓

i=5: binary = 101
  i>>1 = 2  →  dp[2] = 1
  i&1  = 1  (last bit is 1)
  dp[5] = 1 + 1 = 2  ✓
```

## The Picture

```
i   binary   i>>1  dp[i>>1]  i&1  dp[i]
─── ──────── ───── ─────────  ───  ─────
0   000      —     —          —    0
1   001      0     0          1    1
2   010      1     1          0    1
3   011      1     1          1    2
4   100      2     1          0    1
5   101      2     1          1    2
6   110      3     2          0    2
7   111      3     2          1    3
8   1000     4     1          0    1

Pattern: each row reuses a row half its index.
Right-shift ≡ remove the last bit.
Add 1 only if that removed bit was 1.

Visualization of reuse:
  dp[6] borrows from dp[3]  (6 = 110, 3 = 11)
  dp[8] borrows from dp[4]  (8 = 1000, 4 = 100)
```

## When To Use This Pattern

- When building a table of results **0..n** and each answer depends
  on a smaller index, think DP.
- When a number's bit count relates to **half that number**, think
  `i >> 1` as the DP parent.
- When you see **"count set bits for a range"**, think dp-with-shift
  over calling `bin().count('1')` per element.
- When a problem says **"no built-in function, O(n) time"**, think
  bitwise DP recurrence.
- When you need to **reuse prior bit-level sub-problems**, think
  right-shift as the reduction step.

## The Approach

Create a dp array of size n+1, set dp[0] = 0. For each i from 1
to n, right-shift i by one to get its parent (i with last bit
removed), look up that parent's bit count, then add 1 if i's last
bit is set (`i & 1`). Return the full dp array.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    """Run test cases for count_bits."""
    cases = [
        # (n, expected,                    label)
        (0, [0],                            "n=0"),
        (1, [0, 1],                         "n=1"),
        (2, [0, 1, 1],                      "n=2"),
        (5, [0, 1, 1, 2, 1, 2],            "n=5"),
        (7, [0,1,1,2,1,2,2,3],             "n=7"),
        (8, [0,1,1,2,1,2,2,3,1],           "n=8"),
        (4, [0, 1, 1, 2, 1],               "n=4"),
    ]
    passed = 0
    for n, expected, label in cases:
        got = func(n)
        status = "PASSED" if got == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"[{status}] {label}: "
            f"got={got}, expected={expected}"
        )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")

| Dec | Binary |
|-----|--------|
| 1 | 0001 |
| 2 | 0010 |
| 3 | 0011 |
| 4 | 0100 |
| 5 | 0101 |
| 6 | 0110 |
| 7 | 0111 |
| 8 | 1000 |
| 9 | 1001 |
| 10 | 1010 |
| 11 | 1011 |
| 12 | 1100 |
| 13 | 1101 |
| 14 | 1110 |
| 15 | 1111 |
| 16 | 10000 |
| 17 | 10001 |
| 18 | 10010 |

Notice 4, 8, 16 — powers of 2 always have exactly **one** `1` bit.

In [7]:
def count_bits(n: int) -> List[int]:
    """
    dp[0] = 0
    offset = last power of 2 seen so far
    dp[i] = 1 + dp[i - offset]
    
    i - offset strips the leading 1 bit, leaving a smaller
    number whose bit count is already in dp.
    When i hits the next power of 2, update offset.
    """


    dp = [0] * (n + 1)
    offset = 1

    for i in range(1, n + 1):
        if offset * 2 == i:
            offset = i
        dp[i] = 1 + dp[i-offset]
    return dp



print(count_bits(0))  # expected: [0]
print(count_bits(2))  # expected: [0, 1, 1]
print(count_bits(5))  # expected: [0, 1, 1, 2, 1, 2]
print(count_bits(8))  # expected: [0,1,1,2,1,2,2,3,1]
test_harness(count_bits)        


[0]
[0, 1, 1]
[0, 1, 1, 2, 1, 2]
[0, 1, 1, 2, 1, 2, 2, 3, 1]
[PASSED] n=0: got=[0], expected=[0]
[PASSED] n=1: got=[0, 1], expected=[0, 1]
[PASSED] n=2: got=[0, 1, 1], expected=[0, 1, 1]
[PASSED] n=5: got=[0, 1, 1, 2, 1, 2], expected=[0, 1, 1, 2, 1, 2]
[PASSED] n=7: got=[0, 1, 1, 2, 1, 2, 2, 3], expected=[0, 1, 1, 2, 1, 2, 2, 3]
[PASSED] n=8: got=[0, 1, 1, 2, 1, 2, 2, 3, 1], expected=[0, 1, 1, 2, 1, 2, 2, 3, 1]
[PASSED] n=4: got=[0, 1, 1, 2, 1], expected=[0, 1, 1, 2, 1]

Result: 7/7 passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(count_bits)

## Complexity

| Approach           | Time       | Space  | Notes                     |
|:-------------------|:-----------|:-------|:--------------------------|
| Brute (bin+count)  | O(n log n) | O(1)   | O(log i) per element      |
| Brian Kernighan    | O(n log n) | O(1)   | Loop per set bit          |
| **DP + bit shift** | **O(n)**   | **O(1)**| **Optimal — reuses dp** |

## Real World Connection

**AWS / Cloud:** Hamming weight tables (bit counts) power error-
correcting codes in storage systems. Pre-computing dp arrays for
ranges like this avoids per-block bit-scan overhead at S3 scale.

**Citi / Finance:** Risk scoring systems encode feature flags as
bitmasks. Counting active flags per transaction record in O(n)
across millions of trades is exactly this DP pattern applied.

**Data Engineering:** Bloom filters and HyperLogLog sketches rely
on fast popcount (population count) operations. Understanding this
recurrence helps DE engineers reason about sketch accuracy vs.
memory tradeoffs in approximate-query engines like Presto or Spark.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra